In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
BRONZE_PATH = "workspace.case_spark_cvm.bronze_fii_complemento_cvm"
NOME_TABELA  = f"silver_cvm_fii_complemento" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## CVM - Fundos Imobiliarios - Complemento

In [0]:
df_silver_fii_complemento = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento" )

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_fii_complemento = df_silver_fii_complemento.withColumn(
    "CNPJ_FUNDO_CLASSE",
    PipelineConfig.normalizar_cnpj("CNPJ_FUNDO_CLASSE")
)

#### 1.1.2 Retirando dados duplicados

In [0]:
# 1. Chaves que devem ser únicas
chave_negocio = ["cnpj_fundo_classe", "data_referencia"]

# 2. Removemos as duplicatas 
# Como não á uma regra clara de desempate ficamos com a linha com maior patrimonio liquido
df_silver_fii_complemento, df_quarentena_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_silver_fii_complemento,
    chave_negocio=chave_negocio,
    coluna_ordenacao="patrimonio_liquido" 
)

# 3. Salva a sujeira na quarentena
PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_duplicadas, 
    tabela_origem="bronze_fii_complemento_cvm", 
    data_proc=DATA_PROC
)


#### 1.1.3 Retirando dados nulos de Colunas Cores

In [0]:
regras_qualidade = {
    "CNPJ_Fundo_Classe": "not_null",  # Não pode ser vazio (Substitui o dropna)
    "Data_Referencia": "not_null",    # Não pode ser vazio (Substitui o dropna)
    "Total_Numero_Cotistas": "int",         # Não pode conter letras
    "Valor_Ativo": "decimal",  # Não pode conter letras
    "Patrimonio_Liquido": "decimal"   # Não pode conter letras
}

df_silver_fii_complemento, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_silver_fii_complemento,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_fii_complemento_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_silver_fii_complemento = df_silver_fii_complemento.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
df_silver_fii_complemento = df_silver_fii_complemento\
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
    .withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
    .withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
    .withColumn('data_informacao_numero_cotistas', f.col('Data_Informacao_Numero_Cotistas').cast(t.DateType())) \
    .withColumn('total_numero_cotistas', f.col('Total_Numero_Cotistas').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_pessoa_fisica', f.col('Numero_Cotistas_Pessoa_Fisica').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_pessoa_juridica_nao_financeira', f.col('Numero_Cotistas_Pessoa_Juridica_Nao_Financeira').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_banco_comercial', f.col('Numero_Cotistas_Banco_Comercial').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_corretora_distribuidora', f.col('Numero_Cotistas_Corretora_Distribuidora').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outras_pessoas_juridicas_financeira', f.col('Numero_Cotistas_Outras_Pessoas_Juridicas_Financeira').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_investidores_nao_residentes', f.col('Numero_Cotistas_Investidores_Nao_Residentes').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_entidade_aberta_previdencia_complementar', f.col('Numero_Cotistas_Entidade_Aberta_Previdencia_Complementar').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_entidade_fechada_previdencia_complementar', f.col('numero_cotistas_entidade_fechada_previdencia_complementar').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_regime_proprio_previdencia_servidores_publicos', f.col('Numero_Cotistas_Regime_Proprio_Previdencia_Servidores_Publicos').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_sociedade_seguradora_resseguradora', f.col('Numero_Cotistas_Sociedade_Seguradora_Resseguradora').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_sociedade_capitalizacao_arrendamento_mercantil', f.col('Numero_Cotistas_Sociedade_Capitalizacao_Arrendamento_Mercantil').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_fii', f.col('Numero_Cotistas_FII').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outros_fundos', f.col('Numero_Cotistas_Outros_Fundos').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_distribuidores_fundo', f.col('Numero_Cotistas_Distribuidores_Fundo').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outros_tipos', f.col('Numero_Cotistas_Outros_Tipos').cast(t.IntegerType())) \
    .withColumn('valor_ativo', f.col('Valor_Ativo').cast(t.DecimalType(22, 2))) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(22, 2))) \
    .withColumn('cotas_emitidas', f.col('Cotas_Emitidas').cast(t.DecimalType(22, 2))) \
    .withColumn('valor_patrimonial_cotas', f.col('Valor_Patrimonial_Cotas').cast(t.DecimalType(22, 2))) \
    .withColumn('percentual_despesas_taxa_administracao', f.col('Percentual_Despesas_Taxa_Administracao').cast("double")) \
    .withColumn('percentual_despesas_agente_custodiante', f.col('Percentual_Despesas_Agente_Custodiante').cast("double")) \
    .withColumn('percentual_rentabilidade_efetiva_mes', f.col('Percentual_Rentabilidade_Efetiva_Mes').cast("double")) \
    .withColumn('percentual_rentabilidade_patrimonial_mes', f.col('Percentual_Rentabilidade_Patrimonial_Mes').cast("double")) \
    .withColumn('percentual_dividend_yield_mes', f.col('Percentual_Dividend_Yield_Mes').cast("double")) \
    .withColumn('percentual_amortizacao_cotas_mes', f.col('Percentual_Amortizacao_Cotas_Mes').cast("double"))

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["cnpj_fundo_classe", "data_referencia"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_silver_fii_complemento, 
    tabela_destino=SILVER_PATH, 
    chave_negocio=chave_negocio
    )

In [0]:
%sql
select 
    *
from workspace.case_spark_cvm.silver_cvm_fii_complemento
order by data_referencia desc

In [0]:
%sql
select 
    *
from workspace.case_spark_cvm.silver_quarentena